In [ ]:
import glob, shutil, numpy as np, pandas as pd, chromadb
from pathlib import Path
print(chromadb.__version__)

BASE = Path("data")
print(len(list((BASE / "emb_shards").glob("shard_*.npz"))), "shards found")   # expect about 23

shutil.rmtree(BASE / "chroma_full", ignore_errors=True)       # delete the broken copy

cols = ["doc_id", "ticker", "title", "text", "publish_time", "url", "embed_source", "lexrank_summary"]
meta = pd.read_parquet(BASE / "raw/cleaned_data.parquet", columns=cols).set_index("doc_id")

col = chromadb.PersistentClient(path=str(BASE / "chroma_full")).get_or_create_collection(
    "news", metadata={"hnsw:space": "cosine"})

for f in sorted((BASE / "emb_shards").glob("shard_*.npz")):
    z = np.load(f, allow_pickle=True)
    ids = [str(i) for i in z["ids"]]
    emb = z["emb"]
    p = meta.loc[ids].reset_index()
    for i in range(0, len(p), 512):
        b, eb = p.iloc[i:i+512], emb[i:i+512]
        col.upsert(
            ids=b.doc_id.tolist(), embeddings=eb.tolist(), documents=b.text.tolist(),
            metadatas=[{"ticker": r.ticker, "publish_time": int(r.publish_time),
                        "title": r.title, "url": r.url or "",
                        "embed_source": r.embed_source,
                        "lexrank_summary": r.lexrank_summary or ""}
                       for r in b.itertuples()])
    print(f.name, "| count", col.count(), flush=True)

print(col.count(), len(meta))        # should match

1.5.9
21 shards found
shard_000.npz | count 209262
shard_001.npz | count 209262
shard_002.npz | count 209262
shard_003.npz | count 209262
shard_004.npz | count 209262
shard_005.npz | count 209262
shard_006.npz | count 209262
shard_007.npz | count 209262
shard_008.npz | count 209262
shard_009.npz | count 209262
shard_010.npz | count 209262
shard_011.npz | count 209262
shard_012.npz | count 209262
shard_013.npz | count 209262
shard_014.npz | count 209262
shard_015.npz | count 209262
shard_016.npz | count 209262
shard_017.npz | count 209262
shard_018.npz | count 209262
shard_019.npz | count 209262
shard_020.npz | count 209262
209262 209262


In [3]:
import shutil, chromadb

shutil.copytree("data/chroma_full", "data/chroma_full_new")

c = chromadb.PersistentClient(path="data/chroma_full_new").get_collection("news")
print(c.count())                      # should match len(meta)
print(c.get(limit=2, include=["metadatas"])["metadatas"])

209262
[{'title': 'My 6 Largest Portfolio Holdings Heading Into 2024 -- and the Important Investing Lesson I Learned From Each One', 'url': 'https://www.nasdaq.com/articles/my-6-largest-portfolio-holdings-heading-into-2024-and-the-important-investing-lesson-i', 'embed_source': 'summary', 'ticker': 'AAPL', 'lexrank_summary': '3: Apple There\'s little question that Apple (NASDAQ: AAPL) has become one of the most successful companies in history. 6: Nvidia Every investor has one -- the "stock that got away." Should you invest $1,000 in Nvidia right now?', 'publish_time': 1702764000}, {'url': 'https://www.nasdaq.com/articles/brokers-suggest-investing-in-apple-aapl%3A-read-this-before-placing-a-bet', 'ticker': 'AAPL', 'lexrank_summary': "Brokerage Recommendation Trends for AAPL Let's take a look at what these Wall Street heavyweights have to say about Apple (AAPL) before we discuss the reliability of brokerage recommendations and how to use them to your advantage. Is AAPL a Good Investment?"

In [ ]:
%load_ext autoreload
%autoreload 2

from src.config import *                  # paths, constants, ts, DAY, plus the shared imports
from src.loaders import load_all
from src.retrieval import retrieve, count_leaks
from src.sentiment import weekly_sentiment
from src.prices import daily_returns, history_days
from src.events import load_earnings, resolve_events
from src.events import fetch_earnings, load_earnings

e = load_earnings()
ctx = load_all()

docs   = ctx.docs        # articles + FinBERT scores
sent   = ctx.sent        # doc_id -> pos, neg, neu, sentiment
prices = ctx.prices      # daily adjusted closes + ^GSPC
col    = ctx.col         # Chroma collection
model  = ctx.model       # BGE-small embedder
keep   = ctx.tickers     # the 30 tickers

print(docs.shape, prices.shape, col.count(), len(keep))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

docs (209262, 11) | prices (4183, 31) | chroma 209262 | 30 tickers
(209262, 11) (4183, 31) 209262 30


In [18]:
doc_id = "a743c0977fa51127"
out = col.get(ids=[doc_id], include=["documents", "metadatas"])

meta = out["metadatas"][0]
print("TITLE:", meta["title"])
print("DATE :", pd.to_datetime(meta["publish_time"], unit="s"))
print("URL  :", meta["url"])
print("\nARTICLE (stored, capped at 3000 chars):\n", out["documents"][0])
print("\nLEXRANK SUMMARY:\n", meta["lexrank_summary"])

TITLE: My 6 Largest Portfolio Holdings Heading Into 2024 -- and the Important Investing Lesson I Learned From Each One
DATE : 2023-12-16 22:00:00
URL  : https://www.nasdaq.com/articles/my-6-largest-portfolio-holdings-heading-into-2024-and-the-important-investing-lesson-i

ARTICLE (stored, capped at 3000 chars):
 My 6 Largest Portfolio Holdings Heading Into 2024 -- and the Important Investing Lesson I Learned From Each One. After an absolute disaster of a year in 2022, the stock market appears to have turned the corner. Each of the major market indexes has gained more than 20% from their respective trough. Perhaps more importantly, the S&P 500 and the Nasdaq Composite are within striking distance of new highs, which will check the final box marking the start of a new bull market.
Closing out the old and ringing in the new is a great time for examination, and one of the places I start is with my portfolio. A review of my top investments and how they came to be that way can offer valuable

In [19]:
fb_src = docs.merge(docs[["doc_id"]], on="doc_id")   # docs already has embed_source and scores
g = docs.groupby("embed_source").agg(
    n=("sentiment", "size"),
    mean_neutral=("neu", "mean"),
    mean_abs_sentiment=("sentiment", lambda s: s.abs().mean()),
    share_neutral_dominant=("neu", lambda s: (s > 0.7).mean()))
print(g.round(3))

                   n  mean_neutral  mean_abs_sentiment  share_neutral_dominant
embed_source                                                                  
article       148279         0.433               0.455                   0.357
summary        60983         0.456               0.468                   0.387


In [7]:
import os
from typing import Literal
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()  

client = OpenAI(base_url="https://openrouter.ai/api/v1",
                api_key=os.environ["OPENROUTER_API_KEY"])      # no key in the code

class RetrievalInput(BaseModel):
    ticker: str = Field(description="Uppercase stock ticker, e.g. NVDA")
    search_query: str = Field(description="Short semantic search phrase, no dates or instructions")
    start: str = Field(description="Start date, YYYY-MM-DD")
    end: str = Field(description="End date, YYYY-MM-DD")
    as_of: str = Field(description="Knowledge cutoff date, YYYY-MM-DD")
    tasks: list[Literal["trend", "event_impact", "point_lookup"]] = Field(
        description="Every task the question asks for. A question can need more than one.")
    event_hint: str = Field(description="Event phrase such as 'May earnings', or empty string if none")

SYSTEM_PROMPT = f"""Extract structured arguments from the user's financial question.
Rules:
- ticker: map company names to tickers. Valid tickers: {', '.join(keep)}.
- Convert month or quarter ranges to exact dates ('Jan to Jun 2023' -> start 2023-01-01, end 2023-06-30).
- If the user says 'as of <date>', set as_of to that date. Otherwise set as_of equal to end.
- tasks: 'trend' for how sentiment or coverage changed over time, 'event_impact' for how the market
  reacted to an event, 'point_lookup' for what was the latest news at a date. Include all that apply.
- event_hint: the event phrase only (for example 'May earnings'); empty string if none.
- search_query: the financial topic only."""

def extract_params(user_text, model="openrouter/free"):
    schema = RetrievalInput.model_json_schema()
    schema["additionalProperties"] = False
    resp = client.chat.completions.create(
        model=model, temperature=0,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": user_text}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "retrieval_input", "strict": True, "schema": schema}},
        extra_body={"provider": {"require_parameters": True},
                    "plugins": [{"id": "response-healing"}]},
    )
    return RetrievalInput.model_validate_json(resp.choices[0].message.content)

In [8]:
question = ("How did sentiment around NVDA change from Jan to Jun 2023, "
            "and how did the market react to its May earnings? Answer as of 31 May 2023.")
params = extract_params(question)
print(params.model_dump())

{'ticker': 'NVDA', 'search_query': 'sentiment around NVDA Jan to Jun 2023 and May earnings', 'start': '2023-01-01', 'end': '2023-06-30', 'as_of': '2023-05-31', 'tasks': ['trend', 'event_impact', 'point_lookup'], 'event_hint': 'May earnings'}


In [9]:
res = retrieve(params.search_query, params.ticker, params.start, params.end, params.as_of,
               col=col, model=model, sent=sent)

pd.set_option("display.max_colwidth", 75)
print(len(res), "articles retrieved")
print(res[["date", "bucket", "sim", "decay", "score", "pos", "neg", "neu", "sentiment", "title"]]
      .round(3).to_string())
print("\nall on or before as_of:", res.publish_time.max() <= ts(params.as_of) + DAY - 1)
print("mean sentiment of retrieved:", round(res.sentiment.mean(), 3))

15 articles retrieved
          date  bucket    sim  decay  score    pos    neg    neu  sentiment                                                                                                                   title
0   2023-01-04       0  0.737  0.033  0.526  0.035  0.032  0.933      0.003                                                                                    NVDA June 2025 Options Begin Trading
1   2023-01-12       0  0.732  0.039  0.524  0.909  0.065  0.027      0.844                                                                                      Will Nvidia Stock Recover in 2023?
2   2023-01-24       0  0.732  0.052  0.528  0.213  0.739  0.048     -0.526                                                                Nvidia (NVDA) Gains As Market Dips: What You Should Know
3   2023-02-22       1  0.796  0.102  0.588  0.037  0.933  0.030     -0.896  After-Hours Earnings Report for February 22, 2023 : NVDA, PXD, EBAY, GMAB, ANSS, EXR, SUI, ICLR, CTRA, LCID, MOS, ETS

In [32]:
wk = weekly_sentiment(docs, params.ticker, params.start, params.end, params.as_of)
print(wk.round(3).to_string())

   week_start  mean_sentiment  n_docs  low_confidence
0  2022-12-26          -0.703       1            True
1  2023-01-02          -0.135      35           False
2  2023-01-09           0.002      26           False
3  2023-01-16           0.038      33           False
4  2023-01-23           0.048      44           False
5  2023-01-30          -0.005      35           False
6  2023-02-06           0.128      37           False
7  2023-02-13           0.048      56           False
8  2023-02-20          -0.097      95           False
9  2023-02-27           0.094      65           False
10 2023-03-06           0.071      43           False
11 2023-03-13           0.031      56           False
12 2023-03-20          -0.007      83           False
13 2023-03-27           0.201      58           False
14 2023-04-03           0.077      72           False
15 2023-04-10           0.227      67           False
16 2023-04-17          -0.015      71           False
17 2023-04-24           0.09

C:\Users\Krishna\AppData\Local\Temp\ipykernel_32188\4180670314.py:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(wk.round(3).to_string())


In [35]:
from src.events import load_earnings, resolve_events
from src.event_study import event_study

events = resolve_events(prices, params.ticker, params.start, params.end, params.as_of,
                        hint=params.event_hint, earnings=load_earnings())
print(events)

results = [event_study(prices, params.ticker, e["date"], params.as_of) for e in events]
for r in results:
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})

[{'date': Timestamp('2023-05-25 00:00:00'), 'kind': 'earnings'}]
{'ticker': 'NVDA', 'event_date': '2023-05-25', 'status': 'ok', 'alpha': 0.0049, 'beta': 2.0342, 'sigma': 0.0251, 'n_est': 101, 'car_-1_1': 0.2202, 't_-1_1': 5.058, 'p_-1_1': 0.0, 'mktadj_-1_1': 0.2498, 'car_0_5': None, 'note_0_5': 'window_extends_past_as_of'}
